In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Compute growth rates (µ, h^-1) from Biolog PlateReader OD590 data, with strain name mapping.

Usage:
    python compute_growth_rates.py input.xlsx output.xlsx
If run inside Jupyter, unknown args (like --f=...) are ignored.
"""

import math
import re
import sys
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Tunable parameters ----
EPS = 1e-4            # lower bound after background subtraction to enable ln()
PREFERRED_WINDOW = 4  # try 4-point windows first for µ estimation
MIN_WINDOW = 3        # minimum window length

ID_COL = "strain_number"  # in 'strain_names' sheet
NAME_COL = "name"         # in 'strain_names' sheet

def parse_datetime_mixed(series: pd.Series) -> pd.Series:
    """
    Parse a mix of absolute timestamps, time-only strings, and datetime.time objects
    into pandas Timestamps (using a constant base date).
    """
    import datetime as dt
    import re

    parsed = pd.to_datetime(series, errors="coerce")
    mask = parsed.isna()
    if mask.any():
        base = pd.Timestamp("1900-01-01")
        converted = []
        for val in series[mask]:
            td = pd.NaT
            if isinstance(val, dt.time):
                td = pd.Timedelta(
                    hours=val.hour, minutes=val.minute,
                    seconds=val.second, microseconds=val.microsecond
                )
            if td is pd.NaT:
                try:
                    td_try = pd.to_timedelta(val, errors="coerce")
                    if pd.notna(td_try):
                        td = td_try
                except Exception:
                    td = pd.NaT
            if td is pd.NaT and val is not None:
                s = str(val)
                m = re.search(r"(\d{1,2}:\d{2}:\d{2}|\d{1,2}:\d{2}|\d+(?:\.\d+)?)", s)
                if m:
                    frag = m.group(1)
                    td_try = pd.to_timedelta(frag, errors="coerce")
                    if pd.isna(td_try):
                        try:
                            td_try = pd.to_timedelta(float(frag), unit="h")
                        except Exception:
                            td_try = pd.NaT
                    td = td_try
            converted.append(base + td if (isinstance(td, pd.Timedelta) and pd.notna(td)) else pd.NaT)
        parsed.loc[mask] = converted
    return parsed

def best_mu_for_group(g: pd.DataFrame) -> pd.Series:
    g = g.dropna(subset=["hours", "OD_corr"]).sort_values("hours")
    if len(g) < MIN_WINDOW:
        return pd.Series({
            "mu_max": np.nan, "r2": np.nan, "n_points": len(g),
            "window_start_h": np.nan, "window_end_h": np.nan
        })
    x = g["hours"].to_numpy()
    y = np.log(g["OD_corr"].to_numpy())
    best = {"mu_max": -np.inf, "r2": np.nan, "i0": None, "i1": None}
    n = len(x)
    candidate_lengths = [PREFERRED_WINDOW] + [L for L in range(MIN_WINDOW, n + 1) if L != PREFERRED_WINDOW]
    for L in candidate_lengths:
        for i in range(0, n - L + 1):
            j = i + L
            X = x[i:j]; Y = y[i:j]
            Xc = X - X.mean()
            denom = float(Xc @ Xc)
            if denom == 0:
                continue
            slope = float((Xc @ (Y - Y.mean())) / denom)
            y_hat = slope * X + (Y.mean() - slope * X.mean())
            ss_res = float(np.sum((Y - y_hat) ** 2))
            ss_tot = float(np.sum((Y - Y.mean()) ** 2))
            r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan
            better = (
                (slope > best["mu_max"]) or
                (np.isclose(slope, best["mu_max"], rtol=1e-9) and (np.nan_to_num(r2) > np.nan_to_num(best["r2"]))) or
                (np.isclose(slope, best["mu_max"], rtol=1e-9) and np.nan_to_num(r2) == np.nan_to_num(best["r2"])
                 and (L > (best["i1"] - best["i0"] if best["i0"] is not None else -1)))
            )
            if better:
                best = {"mu_max": slope, "r2": r2, "i0": i, "i1": j}
        if math.isfinite(best["mu_max"]) and not math.isnan(best["mu_max"]) and L == PREFERRED_WINDOW:
            break
    if best["i0"] is None:
        return pd.Series({
            "mu_max": np.nan, "r2": np.nan, "n_points": len(g),
            "window_start_h": np.nan, "window_end_h": np.nan
        })
    return pd.Series({
        "mu_max": best["mu_max"],
        "r2": best["r2"],
        "n_points": len(g),
        "window_start_h": x[best["i0"]],
        "window_end_h": x[best["i1"] - 1]
    })

def aggregate_mu(g: pd.DataFrame) -> pd.Series:
    mu = g["mu_max"].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    n = int(mu.shape[0])
    if n == 0:
        return pd.Series({
            "n_reps": 0, "mu_mean": np.nan, "mu_sd": np.nan, "mu_se": np.nan,
            "mu_ci95_low": np.nan, "mu_ci95_high": np.nan, "doubling_time_h": np.nan,
            "formatted_mean_pm_sd": np.nan, "formatted_mean_pm_se": np.nan
        })
    mean = float(mu.mean())
    sd = float(mu.std(ddof=1)) if n > 1 else 0.0
    se = float(sd / np.sqrt(n)) if n > 1 else np.nan
    if n > 1:
        from scipy.stats import t
        tval = float(t.ppf(0.975, df=n - 1))
        ci_low = mean - tval * se
        ci_high = mean + tval * se
    else:
        ci_low = ci_high = np.nan
    dt = float(np.log(2) / mean) if mean > 0 else np.nan
    def dyn_fmt(x: float) -> str:
        if pd.isna(x):
            return "NA"
        return f"{x:.3f}" if abs(x) < 1 else f"{x:.2f}"
    formatted_sd = f"{dyn_fmt(mean)}±{dyn_fmt(sd)}"
    formatted_se = f"{dyn_fmt(mean)}±{dyn_fmt(se)}" if n > 1 else f"{dyn_fmt(mean)}±NA"
    return pd.Series({
        "n_reps": n, "mu_mean": mean, "mu_sd": sd, "mu_se": se,
        "mu_ci95_low": ci_low, "mu_ci95_high": ci_high, "doubling_time_h": dt,
        "formatted_mean_pm_sd": formatted_sd, "formatted_mean_pm_se": formatted_se
    })

def main(input_xlsx: str, output_xlsx: str) -> None:
    # ---- Load data ----
    input_path = Path(input_xlsx)
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_xlsx}")
    od = pd.read_excel(input_path, sheet_name="OD590")
    sn = pd.read_excel(input_path, sheet_name="strain_names")

    # ---- Time to hours per strain ----
    od["__dt"] = parse_datetime_mixed(od["time"])
    if "strain" not in od.columns:
        od["strain"] = "unknown_strain"
    od = od.sort_values(["strain", "__dt"]).copy()
    od["hours"] = od.groupby("strain")["__dt"].transform(lambda x: (x - x.min()).dt.total_seconds() / 3600.0)

    # ---- Identify measurement columns and coerce numeric ----
    meta_cols = {"strain", "time", "timepoint", "__dt", "hours"}
    meas_cols = [c for c in od.columns if c not in meta_cols]
    for c in meas_cols:
        od[c] = pd.to_numeric(od[c], errors="coerce")

    # ---- Background subtraction (water mean at each row) ----
    water_cols = [c for c in meas_cols if str(c).startswith("water_")]
    non_water_cols = [c for c in meas_cols if c not in water_cols]
    if water_cols:
        od["__mean_water"] = od[water_cols].mean(axis=1, skipna=True)
    else:
        od["__mean_water"] = np.nan

    corrected = od.copy()
    for c in non_water_cols:
        corrected[c] = (corrected[c] - corrected["__mean_water"]).clip(lower=EPS)

    # ---- Long-format with replicate parsing ----
    rows = []
    rep_pat = re.compile(r"^(.*)_(\d+)$")
    for c in non_water_cols:
        m = rep_pat.match(str(c))
        if m:
            base, rep = m.group(1), int(m.group(2))
        else:
            base, rep = str(c), 1
        tmp = corrected[["strain", "hours", "timepoint"]].copy()
        tmp["substrate"] = base
        tmp["replicate"] = rep
        tmp["OD_corr"] = corrected[c].astype(float)
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)

    # ---- Map strain IDs → names ----
    if ID_COL not in sn.columns or NAME_COL not in sn.columns:
        raise ValueError(f"'strain_names' must contain '{ID_COL}' and '{NAME_COL}'. Found: {list(sn.columns)}")
    long_df = long_df.merge(sn[[ID_COL, NAME_COL]], left_on="strain", right_on=ID_COL, how="left")
    long_df.rename(columns={NAME_COL: "strain_name"}, inplace=True)
    long_df["strain_label"] = long_df["strain_name"].fillna(long_df["strain"])

    # ---- Per-replicate µ ----
    per_rep = long_df.groupby(["strain_label", "substrate", "replicate"], as_index=False).apply(best_mu_for_group)
    if isinstance(per_rep.index, pd.MultiIndex):
        per_rep = per_rep.reset_index(level=3, drop=True).reset_index()

    # ---- Aggregate across replicates ----
    per_group = per_rep.groupby(["strain_label", "substrate"], as_index=False).apply(aggregate_mu)
    if isinstance(per_group.index, pd.MultiIndex):
        per_group = per_group.reset_index(level=2, drop=True).reset_index()

    # ---- Save Excel ----
    output_path = Path(output_xlsx)
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        long_df.to_excel(writer, sheet_name="long_data", index=False)
        per_rep.to_excel(writer, sheet_name="per_replicate_mu", index=False)
        per_group.to_excel(writer, sheet_name="per_strain_substrate", index=False)

    print(f"Wrote: {output_path.resolve()}")

if __name__ == "__main__":
    default_input = "Senka_Biolog_PlateReader_allstrainscombinedIsaline.xlsx"
    default_output = "growth_rates_with_names.xlsx"
    parser = argparse.ArgumentParser(add_help=True)
    parser.add_argument("input", nargs="?", default=default_input)
    parser.add_argument("output", nargs="?", default=default_output)
    args, _unknown = parser.parse_known_args()
    in_arg = args.input
    if isinstance(in_arg, str) and in_arg.startswith("-"):
        in_arg = default_input
        print("Note: Ignoring unexpected option-like input argument; using default input:", in_arg)
    try:
        main(in_arg, args.output)
    except FileNotFoundError as e:
        print(e)
        print("\nTip: In Jupyter, you can also do:\n"
              "    from compute_growth_rates import main\n"
              f"    main('{default_input}', '{default_output}')")
